# Gymnasium single-symbol environment design
Milestone 5A validates the production `single_symbol_env_v1` simulator. At step *t*, the agent sees data through date *t*, trades at date *t+1* open, and is valued at date *t+1* close. This notebook does not train PPO.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)

Project root: /Users/m.abdulbasit/Downloads/virtual-trader
Python executable: /Users/m.abdulbasit/Downloads/virtual-trader/.venv/bin/python


## Load one configurable eligible symbol dataset

In [2]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from data_pipeline.src.config import AI_MINIMUM_USABLE_ROWS, PROCESSED_SYMBOLS_DIR
from reinforcement_learning.environments import SingleSymbolEnvConfig, SingleSymbolTradingEnv
from reinforcement_learning.environments.config import DEFAULT_OBSERVATION_FEATURES
from reinforcement_learning.environments.validation import validate_environment
from reinforcement_learning.evaluation import BuyAndHoldPolicy, RandomPolicy, run_baseline

preferred_symbol = 'MCB'
paths = sorted(PROCESSED_SYMBOLS_DIR.glob('*.csv'))
eligible = []
for path in paths:
    candidate = pd.read_csv(path, dtype={'symbol': 'string'})
    if len(candidate) >= AI_MINIMUM_USABLE_ROWS:
        eligible.append((path, candidate))
selected = next(((p, d) for p, d in eligible if p.stem == preferred_symbol), eligible[0] if eligible else None)
demonstration_mode = selected is None
if demonstration_mode:
    display(Markdown('### ⚠️ Demonstration mode — no local symbol meets the configured readiness threshold. This deterministic fixture validates mechanics only and is **not suitable for research conclusions**.'))
    rows = 80
    dates = pd.bdate_range('2025-01-01', periods=rows)
    close = 100 + np.linspace(0, 12, rows) + np.sin(np.arange(rows) / 4)
    dataset = pd.DataFrame({'symbol': 'DEMO', 'date': dates, 'open': close - 0.3, 'high': close + 1, 'low': close - 1, 'close': close, 'volume': 100_000 + np.arange(rows) * 100})
    for position, column in enumerate(DEFAULT_OBSERVATION_FEATURES): dataset[column] = np.sin(np.arange(rows) / (position + 2))
else:
    dataset_path, dataset = selected
    display(Markdown(f'### Production dataset: {dataset_path.stem}'))
print('Rows:', len(dataset), 'Range:', dataset['date'].min(), 'to', dataset['date'].max())
print('Feature columns:', list(DEFAULT_OBSERVATION_FEATURES))

### Production dataset: MCB

Rows: 2435 Range: 2016-10-06 to 2026-08-05
Feature columns: ['simple_return', 'log_return', 'high_low_range', 'open_close_return', 'rolling_volatility_20', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'obv', 'volume_ma_20']


## Configure and instantiate the production environment

In [3]:
config = SingleSymbolEnvConfig()
env = SingleSymbolTradingEnv(dataset, config)
print('Version:', config.environment_version)
print('Action space:', env.action_space)
print('Observation space:', env.observation_space)
print('Observation features:', env.observation_feature_names)

Version: single_symbol_env_v1
Action space: Discrete(3)
Observation space: Box(-3.4028235e+38, 3.4028235e+38, (17,), float32)
Observation features: ('simple_return', 'log_return', 'high_low_range', 'open_close_return', 'rolling_volatility_20', 'rsi_14', 'macd', 'macd_signal', 'macd_histogram', 'atr_14', 'obv', 'volume_ma_20', 'portfolio_cash_ratio', 'portfolio_position_value_ratio', 'portfolio_position_indicator', 'portfolio_unrealized_return_ratio', 'portfolio_current_drawdown')


## Reset and inspect one observation

In [4]:
observation, reset_info = env.reset(seed=42)
display(pd.Series(observation, index=env.observation_feature_names, name='Value'))
display(reset_info)

simple_return                       -1.331593e-02
log_return                          -1.340538e-02
high_low_range                       4.990000e+00
open_close_return                   -1.331593e-02
rolling_volatility_20                1.307156e-02
rsi_14                               6.634064e+01
macd                                 3.400937e+00
macd_signal                          1.462883e+00
macd_histogram                       1.938054e+00
atr_14                               5.277857e+00
obv                                  6.570900e+06
volume_ma_20                         1.180045e+06
portfolio_cash_ratio                 1.000000e+00
portfolio_position_value_ratio       0.000000e+00
portfolio_position_indicator         0.000000e+00
portfolio_unrealized_return_ratio    0.000000e+00
portfolio_current_drawdown           0.000000e+00
Name: Value, dtype: float32

{'environment_version': 'single_symbol_env_v1',
 'date': Timestamp('2016-10-06 00:00:00'),
 'next_date': Timestamp('2016-10-07 00:00:00'),
 'action': None,
 'action_name': None,
 'execution_price': None,
 'shares_traded': 0,
 'transaction_cost': 0.0,
 'cash': 1000000.0,
 'shares_held': 0,
 'portfolio_value': 1000000.0,
 'realized_profit_loss': 0.0,
 'unrealized_profit_loss': 0.0,
 'drawdown': 0.0,
 'reward_components': {'portfolio_growth': 0.0,
  'transaction_cost_penalty': 0.0,
  'drawdown_penalty': 0.0,
  'invalid_action_penalty': 0.0}}

## Manual Buy, Hold, Sell transitions

In [5]:
env.reset(seed=42)
manual_info = []
for action in (1, 0, 2):
    _, reward, terminated, truncated, info = env.step(action)
    manual_info.append(info)
display(manual_info)
display(env.get_history())

[{'environment_version': 'single_symbol_env_v1',
  'date': Timestamp('2016-10-06 00:00:00'),
  'next_date': Timestamp('2016-10-07 00:00:00'),
  'action': 1,
  'action_name': 'Buy',
  'execution_price': 228.11399999999998,
  'shares_traded': 4379,
  'transaction_cost': 1498.1172059998944,
  'cash': 89.88279400009196,
  'shares_held': 4379,
  'portfolio_value': 979453.2327940001,
  'realized_profit_loss': 0.0,
  'unrealized_profit_loss': -20546.76720599993,
  'drawdown': 0.020546767205999933,
  'reward_components': {'portfolio_growth': -0.020760788736397074,
   'transaction_cost_penalty': -0.0,
   'drawdown_penalty': -0.0020546767205999934,
   'invalid_action_penalty': -0.0}},
 {'environment_version': 'single_symbol_env_v1',
  'date': Timestamp('2016-10-07 00:00:00'),
  'next_date': Timestamp('2016-10-10 00:00:00'),
  'action': 0,
  'action_name': 'Hold',
  'execution_price': None,
  'shares_traded': 0,
  'transaction_cost': 0.0,
  'cash': 89.88279400009196,
  'shares_held': 4379,
  'por

,initial_portfolio_value,observation_date,execution_date,action,action_name,execution_price,shares_traded,transaction_cost,cash,shares_held,portfolio_value,realized_profit_loss,unrealized_profit_loss,drawdown,reward
0,1000000.0,2016-10-06,2016-10-07,1,Buy,228.1140,4379,1498.117206,89.882794,4379,979453.232794,0.000000,-20546.767206,0.020547,-0.022815
1,1000000.0,2016-10-07,2016-10-10,0,Hold,NaN,0,0.000000,89.882794,4379,983963.602794,0.000000,-16036.397206,0.016036,0.004594
2,1000000.0,2016-10-10,2016-10-13,2,Sell,224.8875,-4379,1477.419862,983887.462932,0,983887.462932,-16112.537068,0.000000,0.016113,-0.000085


## Complete deterministic baselines

In [6]:
buy_hold = run_baseline(SingleSymbolTradingEnv(dataset, config), BuyAndHoldPolicy(), seed=42)
random_run = run_baseline(SingleSymbolTradingEnv(dataset, config), RandomPolicy(seed=42), seed=42)
comparison = pd.DataFrame([
    {'Baseline': 'Buy and Hold', **{k: v for k, v in buy_hold.metrics.items() if k != 'daily_returns'}},
    {'Baseline': 'Fixed-seed Random', **{k: v for k, v in random_run.metrics.items() if k != 'daily_returns'}},
])
display(comparison)
display(buy_hold.history)

,Baseline,initial_portfolio_value,final_portfolio_value,total_return,maximum_drawdown,number_of_trades,total_transaction_costs,sharpe_ratio,annualized_volatility
0,Buy and Hold,1000000.0,1.826658e+06,0.826658,0.589576,1,1498.117206,0.368244,0.263559
1,Fixed-seed Random,1000000.0,6.453372e+05,-0.354663,0.647643,789,744315.099136,-0.136103,0.194373


,initial_portfolio_value,observation_date,execution_date,action,action_name,execution_price,shares_traded,transaction_cost,cash,shares_held,portfolio_value,realized_profit_loss,unrealized_profit_loss,drawdown,reward
0,1000000.0,2016-10-06,2016-10-07,1,Buy,228.114,4379,1498.117206,89.882794,4379,9.794532e+05,0.0,-20546.767206,0.020547,-0.022815
1,1000000.0,2016-10-07,2016-10-10,0,Hold,NaN,0,0.000000,89.882794,4379,9.839636e+05,0.0,-16036.397206,0.016036,0.004594
2,1000000.0,2016-10-10,2016-10-13,0,Hold,NaN,0,0.000000,89.882794,4379,9.755559e+05,0.0,-24444.077206,0.024444,-0.009422
3,1000000.0,2016-10-13,2016-10-14,0,Hold,NaN,0,0.000000,89.882794,4379,9.892622e+05,0.0,-10737.807206,0.010738,0.013952
4,1000000.0,2016-10-14,2016-10-17,0,Hold,NaN,0,0.000000,89.882794,4379,9.786212e+05,0.0,-21378.777206,0.021379,-0.011879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2429,1000000.0,2026-07-29,2026-07-30,0,Hold,NaN,0,0.000000,89.882794,4379,1.782299e+06,0.0,782299.092794,0.089667,0.000270
2430,1000000.0,2026-07-30,2026-07-31,0,Hold,NaN,0,0.000000,89.882794,4379,1.789656e+06,0.0,789655.812794,0.085909,0.004119
2431,1000000.0,2026-07-31,2026-08-03,0,Hold,NaN,0,0.000000,89.882794,4379,1.786065e+06,0.0,786065.032794,0.087743,-0.002192
2432,1000000.0,2026-08-03,2026-08-04,0,Hold,NaN,0,0.000000,89.882794,4379,1.778489e+06,0.0,778489.362794,0.091613,-0.004638


## Portfolio-value plot

In [7]:
plot_data = pd.concat([buy_hold.history.assign(Baseline='Buy and Hold'), random_run.history.assign(Baseline='Fixed-seed Random')])
alt.Chart(plot_data).mark_line().encode(x=alt.X('execution_date:T', title='Trading Date'), y=alt.Y('portfolio_value:Q', title='Portfolio Value (PKR)'), color='Baseline:N', tooltip=['Baseline', 'execution_date:T', 'portfolio_value:Q']).properties(width=750, height=350)

alt.Chart(...)

## Leakage and accounting sanity checks

In [8]:
validation = validate_environment(SingleSymbolTradingEnv(dataset, config))
assert validation.valid, validation.errors
history = buy_hold.history
assert (history['observation_date'] < history['execution_date']).all()
assert np.allclose(history['portfolio_value'], history['cash'] + history['shares_held'] * dataset.set_index(pd.to_datetime(dataset['date'])).loc[pd.to_datetime(history['execution_date']), 'close'].to_numpy())
assert (history['cash'] >= 0).all() and (history['shares_held'] >= 0).all()
display(validation)

EnvironmentValidationResult(environment_version='single_symbol_env_v1', valid=True, observation_shape=(17,), errors=())

## Limitations and PPO readiness
Environment v1 is a deterministic single-symbol, long-only, all-in/all-out simulator. It excludes short selling, leverage, fractional shares, exact broker fee schedules, liquidity/market-impact modelling, corporate actions, and multi-asset allocation. Baselines are not AI models. PPO training, tuning, walk-forward evaluation, and registry writes are deferred to Milestone 5B. Demonstration-fixture results are mechanical checks only.